## Redes Neurais Artificiais 2026.2

- **Disciplina**: Redes Neurais Artificiais 2026.2
- **Professora**: Elloá B. Guedes (ebgcosta@uea.edu.br)  
- **Github**: http://github.com/elloa  
        

Levando em conta a base de dados **_Dry Bean Dataset_**, esta segunda parte do Projeto Prático 2.2 diz respeito à proposição e avaliação de múltiplas redes neurais artificiais do tipo Feedforward Multilayer Perceptron para o problema da classificação multiclasse de variedades de feijão seco a partir de características geométricas extraídas de imagens dos grãos.

## Testando Redes Neurais com os Atributos Numéricos

1. Abra a base de dados em questão
2. Verifique que todos os atributos preditores são numéricos. O único atributo categórico é o alvo (`Class`), que não deve entrar em X
3. Armazene o atributo alvo em uma variável y e os atributos preditores em uma variável X
4. Efetue uma partição holdout 70/30 com o sklearn, distribuindo os exemplos de maneira aleatória e estratificada (`stratify=y`), com `random_state` fixo para reprodutibilidade
5. Efetue o escalonamento dos atributos

In [ ]:
## Reservado para a importação de bibliotecas
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, f1_score, precision_score, recall_score, confusion_matrix
from sklearn import preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier


In [13]:
df = pd.read_excel('DryBeanDataset/Dry_Bean_Dataset.xlsx')

x = df.loc[ : , df.columns != 'Class']
y = df['Class']

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.30, random_state=13, stratify=y)

x_train_stand = x_train.copy()
x_test_stand = x_test.copy()

for column in x_train:
    
    # fit on training data column
    scale = StandardScaler().fit(x_train_stand[[column]])
    
    # transform the training data column
    x_train_stand[column] = scale.transform(x_train_stand[[column]])
    
    # transform the testing data column
    x_test_stand[column] = scale.transform(x_test_stand[[column]])


### Escalonando os atributos

O treinamento de uma rede neural artificial é mais eficiente quando os valores que lhes são fornecidos como entrada são pequenos e estão em escalas semelhantes, pois isto favorece a convergência. Isto é feito escalonando-se os atributos, mas precisa ser feito de maneira cautelosa, para que informações do conjunto de teste não sejam fornecidas no treinamento.

Há duas estratégias para tal escalonamento: normalização (que leva os valores ao intervalo [0,1]) e padronização (que centraliza os dados em média 0 e desvio padrão 1). Ambas possuem características particulares, vantagens e limitações, como é possível ver aqui: https://www.analyticsvidhya.com/blog/2020/04/feature-scaling-machine-learning-normalization-standardization/

No nosso caso, vamos usar a padronização. Assim, com os atributos preditores do treinamento, isto é, X_train, deve-se subtrair a média e dividir pelo desvio padrão. Para tanto, use o StandardScaler e ajuste-o com os dados do conjunto de treino (fit), transformando-os em seguida (transform). Depois, apenas transforme os dados do conjunto de teste (transform), usando a média e o desvio padrão calculados no treino, sem chamar o fit novamente.

Se todo o conjunto X for utilizado na padronização, a rede neural receberá informações do conjunto de teste por meio da média e do desvio padrão utilizados para preparar os dados de treinamento, o que não é desejável (vazamento de dados, ou *data leakage*).


### Continuando

1. Treine uma rede neural multilayer perceptron para este problema com uma única camada oculta de dez neurônios 
    1.1 Utilize a função de ativação ReLU  
    1.2 Utilize o solver Adam    
    1.3 Imprima o passo a passo do treinamento 
    1.4 Utilize o número máximo de épocas igual a 300
2. Com o modelo em questão, após o treinamento, apresente os resultados para o conjunto de teste:  
    2.1 Matriz de confusão  
    2.2 Acurácia  
    2.3 F-Score (F1)  
    2.4 Precisão  
    2.5 Revocação   
3. Repita o treinamento da mesma rede anterior sem imprimir o passo a passo por 10 vezes  
    3.1 Cada uma destas repetições deve ser feita com uma nova partição holdout 70/30 estratificada, e o escalonamento deve ser reajustado com os dados de treino de cada partição  
    3.2 Apresente a média e o desvio padrão da acurácia e do F-Score para o conjunto de teste  
4. Repita por 10 vezes o treinamento desta mesma rede, mas utilizando o otimizador SGD, com as mesmas 10 partições do item anterior  
    4.1 Apresente a média e o desvio padrão da acurácia e do F-Score para o conjunto de teste  
5. Houve influência da escolha do otimizador no desempenho da rede? Justifique com as médias e os desvios padrão obtidos

In [ ]:
# 1) Treinando uma rede com 10 neurônios na camada oculta, ReLU, Adam, 300 epocas
mlp = MLPClassifier(hidden_layer_sizes=(10,), activation='relu', solver='adam',
                     max_iter=300, random_state=0, verbose=True)
mlp.fit(x_train_stand, y_train)


In [ ]:
# 2) Resultados no conjunto de teste
y_pred = mlp.predict(x_test_stand)

print('Matriz de confusão:')
print(confusion_matrix(y_test, y_pred))
print()
print('Acurácia:', accuracy_score(y_test, y_pred))
print('F-Score (macro):', f1_score(y_test, y_pred, average='macro'))
print('Precisão (macro):', precision_score(y_test, y_pred, average='macro'))
print('Revocação (macro):', recall_score(y_test, y_pred, average='macro'))


In [ ]:
# 3) e 4) Repetindo o treinamento 10 vezes, cada vez com uma nova particao holdout 70/30

particoes = []
for i in range(10):
    tr_idx, te_idx = train_test_split(x.index, test_size=0.30, random_state=i, stratify=y)
    particoes.append((tr_idx, te_idx))

resultados_base = []
for solver in ['adam', 'sgd']:
    for p, (tr_idx, te_idx) in enumerate(particoes):
        x_tr, x_te = x.loc[tr_idx], x.loc[te_idx]
        y_tr, y_te = y.loc[tr_idx], y.loc[te_idx]

        escalonador = StandardScaler().fit(x_tr)
        x_tr_s = escalonador.transform(x_tr)
        x_te_s = escalonador.transform(x_te)

        rede = MLPClassifier(hidden_layer_sizes=(10,), activation='relu', solver=solver,
                              max_iter=300, random_state=p)
        rede.fit(x_tr_s, y_tr)
        pred = rede.predict(x_te_s)

        resultados_base.append({
            'solver': solver,
            'particao': p,
            'acuracia': accuracy_score(y_te, pred),
            'f1_macro': f1_score(y_te, pred, average='macro'),
        })

resultados_base = pd.DataFrame(resultados_base)
resumo_base = resultados_base.groupby('solver')[['acuracia', 'f1_macro']].agg(['mean', 'std'])
print(resumo_base)


        acuracia            f1_macro
            mean       std      mean       std
solver
adam    0.929016  0.003869  0.940240  0.003262
sgd     0.924706  0.004630  0.936235  0.003758


**5) Houve influência da escolha do otimizador?**

Sim, mas de forma discreta. Com Adam, a acurácia média das 10 repetições foi de aproximadamente
**0,929 ± 0,004** e o F-Score macro foi de **0,940 ± 0,003**. Com SGD, a acurácia média foi de
**0,925 ± 0,005** e o F-Score macro foi de **0,936 ± 0,004**. Ou seja, o Adam apresentou desempenho
médio um pouco superior ao SGD (cerca de 0,4 a 0,5 ponto percentual) e também um desvio padrão
levemente menor, o que indica resultados um pouco mais estáveis entre as repetições. Isto é esperado,
pois o Adam ajusta a taxa de aprendizado de forma adaptativa durante o treinamento, favorecendo uma
convergência mais rápida e consistente do que o SGD "puro" (taxa de aprendizado fixa), especialmente
dentro de um número limitado de épocas.

## Discussão

Nos passos anteriores, você avaliou o desempenho de uma única arquitetura de rede neural, com uma única camada oculta de 10 neurônios e função de ativação ReLU. O otimizador utilizado, seja o SGD ou o Adam, é o algoritmo que usa o gradiente do erro (calculado pela retropropagação) para atualizar os pesos da rede. Neste sentido, a escolha do otimizador é um hiperparâmetro: ela é definida antes do treinamento, não é aprendida a partir dos dados e controla como a rede será treinada. Também são hiperparâmetros a quantidade de camadas ocultas, o número de neurônios, a função de ativação, a quantidade de épocas, a taxa de aprendizado inicial, dentre outros. Já os parâmetros da rede são os pesos e os vieses, ajustados durante o treinamento.

Cabe salientar também que você efetuou o treinamento desta rede por 10 vezes e apresentou os resultados em termos de média ± desvio padrão. Lembre-se que em uma rede neural há a inicialização aleatória de pesos e, em consequência, o desempenho está sujeito a uma flutuação estocástica. Além disso, cada repetição utilizou uma nova partição holdout, o que introduz uma segunda fonte de variação. A execução múltipla das repetições reduz a influência de uma boa ou má "sorte" na escolha dos pesos iniciais e da partição, que poderia distorcer a conclusão no caso de uma única execução, e permite estimar a variabilidade do desempenho por meio do desvio padrão.

Você também aprendeu uma estratégia para escalonar os atributos para uma melhor convergência da rede. Utilize-a em todos os treinamentos e testes propostos a seguir.

## Propondo Novas Arquiteturas

Variando a arquitetura da rede (uma ou duas camadas ocultas, com diferentes números de neurônios em cada uma delas, e a função de ativação) e os hiperparâmetros solver (Adam ou SGD) e número máximo de épocas (100, 150 e 200), atenda ao que se pede:

1. Proponha 10 arquiteturas distintas de RNAs para o problema em questão, à sua escolha
2. Avalie cada uma das arquiteturas perante todas as combinações de hiperparâmetros (2 solvers × 3 números de épocas), repetindo cada treinamento/avaliação por 10 vezes  
    2.1 Cada repetição deve utilizar uma nova partição holdout 70/30 estratificada, com o escalonamento ajustado apenas com os dados de treino, e as mesmas 10 partições devem ser usadas em todas as configurações, para que a comparação seja justa  
    2.2 Ao todo, serão 60 configurações (10 arquiteturas × 2 solvers × 3 números de épocas) e 600 treinamentos
3. Como resultado da avaliação, apresente, com base no conjunto de teste:  
    3.1 As 3 melhores configurações (arquitetura + solver + número de épocas) no tocante ao F-Score (média macro) e à acurácia, considerando a média das 10 repetições (± desvio padrão)  
    3.2 Para cada uma dessas 3 configurações, a repetição em que houve o melhor desempenho: apresente a matriz de confusão e ilustre, para cada classe, os valores de VP (verdadeiros positivos), VN (verdadeiros negativos), FP (falsos positivos) e FN (falsos negativos)

In [ ]:
# Propondo 10 arquiteturas próprias, testadas com os solvers Adam e SGD e com max_iter = 100, 150 e 200, usando as mesmas 10 partições holdout definidas anteriormente.

arquiteturas_proprias = {
    'A1': (5,), 'A2': (10,), 'A3': (15,), 'A4': (20,), 'A5': (25,),
    'A6': (32,), 'A7': (40,), 'A8': (50,), 'A9': (64,), 'A10': (100,),
}

linhas = []
for nome_arq, camadas in arquiteturas_proprias.items():
    for solver in ['adam', 'sgd']:
        for epocas in [100, 150, 200]:
            for p, (tr_idx, te_idx) in enumerate(particoes):
                x_tr, x_te = x.loc[tr_idx], x.loc[te_idx]
                y_tr, y_te = y.loc[tr_idx], y.loc[te_idx]

                escalonador = StandardScaler().fit(x_tr)
                x_tr_s = escalonador.transform(x_tr)
                x_te_s = escalonador.transform(x_te)

                rede = MLPClassifier(hidden_layer_sizes=camadas, activation='relu',
                                      solver=solver, max_iter=epocas, random_state=p)
                rede.fit(x_tr_s, y_tr)
                pred = rede.predict(x_te_s)

                linhas.append({
                    'arquitetura': nome_arq, 'camadas': camadas, 'solver': solver,
                    'epocas': epocas, 'particao': p,
                    'acuracia': accuracy_score(y_te, pred),
                    'f1_macro': f1_score(y_te, pred, average='macro'),
                })

resultados_proprios = pd.DataFrame(linhas)
print('Quantidade de treinamentos realizados:', len(resultados_proprios))


Quantidade de treinamentos realizados: 600


In [ ]:
# Resumo por configuração e top-3 segundo F-Score e Acurácia
resumo_proprios = resultados_proprios.groupby(['arquitetura', 'camadas', 'solver', 'epocas']).agg(
    acuracia_media=('acuracia', 'mean'), acuracia_dp=('acuracia', 'std'),
    f1_media=('f1_macro', 'mean'), f1_dp=('f1_macro', 'std'),
).reset_index()

print('Top-3 configurações por F-Score (média macro):')
print(resumo_proprios.sort_values('f1_media', ascending=False).head(3).to_string(index=False))
print()
print('Top-3 configurações por Acurácia:')
print(resumo_proprios.sort_values('acuracia_media', ascending=False).head(3).to_string(index=False))


Top-3 configurações por F-Score (média macro):
arquitetura camadas solver  epocas  acuracia_media  acuracia_dp  f1_media    f1_dp
         A9   (64,)   adam     200        0.934231     0.004251  0.945330  0.003566
        A10  (100,)   adam     150        0.933668     0.004495  0.944821  0.003620
         A8   (50,)   adam     200        0.933080     0.003703  0.943988  0.003414

Top-3 configurações por Acurácia:
arquitetura camadas solver  epocas  acuracia_media  acuracia_dp  f1_media    f1_dp
         A9   (64,)   adam     200        0.934231     0.004251  0.945330  0.003566
        A10  (100,)   adam     150        0.933668     0.004495  0.944821  0.003620
        A10  (100,)   adam     200        0.933105     0.004437  0.944592  0.003579


In [ ]:
# Para as 3 melhores configuracoes, localizamos a partição com melhor F-Score no conjunto de teste e apresentamos a matriz de confusão e VP/VN/FP/FN por classe.

melhores_proprias = [('A9', (64,), 'adam', 200), ('A10', (100,), 'adam', 150), ('A8', (50,), 'adam', 200)]
classes = sorted(y.unique())

for nome_arq, camadas, solver, epocas in melhores_proprias:
    melhor = None
    for p, (tr_idx, te_idx) in enumerate(particoes):
        x_tr, x_te = x.loc[tr_idx], x.loc[te_idx]
        y_tr, y_te = y.loc[tr_idx], y.loc[te_idx]
        escalonador = StandardScaler().fit(x_tr)
        x_tr_s = escalonador.transform(x_tr)
        x_te_s = escalonador.transform(x_te)
        rede = MLPClassifier(hidden_layer_sizes=camadas, activation='relu', solver=solver,
                              max_iter=epocas, random_state=p)
        rede.fit(x_tr_s, y_tr)
        pred = rede.predict(x_te_s)
        f1 = f1_score(y_te, pred, average='macro')
        if melhor is None or f1 > melhor['f1']:
            melhor = {'particao': p, 'f1': f1, 'cm': confusion_matrix(y_te, pred, labels=classes)}

    print(f'--- {nome_arq} {camadas} | solver={solver} | épocas={epocas} | melhor particao={melhor["particao"]} | F1={melhor["f1"]:.4f} ---')
    cm = melhor['cm']
    print(pd.DataFrame(cm, index=classes, columns=classes))
    total = cm.sum()
    for i, c in enumerate(classes):
        vp = cm[i, i]
        fp = cm[:, i].sum() - vp
        fn = cm[i, :].sum() - vp
        vn = total - vp - fp - fn
        print(f'  {c}: VP={vp}  VN={vn}  FP={fp}  FN={fn}')
    print()


--- A9 (64,) | solver=adam | épocas=200 | melhor partição=5 | F1=0.9523 ---
          BARBUNYA  BOMBAY  CALI  DERMASON  HOROZ  SEKER  SIRA
BARBUNYA       373       0    16         0      1      2     5
BOMBAY           0     157     0         0      0      0     0
CALI            12       0   467         0      3      1     6
DERMASON         0       0     0      1005      2      6    51
HOROZ            1       0     7         4    560      0     6
SEKER            3       0     0        14      1    579    11
SIRA             5       0     3        52     10      8   713
  BARBUNYA: VP=373  VN=3666  FP=21  FN=24
  BOMBAY: VP=157  VN=3927  FP=0  FN=0
  CALI: VP=467  VN=3569  FP=26  FN=22
  DERMASON: VP=1005  VN=2950  FP=70  FN=59
  HOROZ: VP=560  VN=3489  FP=17  FN=18
  SEKER: VP=579  VN=3459  FP=17  FN=29
  SIRA: VP=713  VN=3214  FP=79  FN=78

--- A10 (100,) | solver=adam | épocas=150 | melhor partição=5 | F1=0.9510 ---
          BARBUNYA  BOMBAY  CALI  DERMASON  HOROZ  SEKER  SIRA
B

## Estimando o número de neurônios

Um dos problemas de pesquisa com redes neurais artificiais consiste na determinação do número de neurônios em sua arquitetura. Embora não seja possível definir a priori qual rede neural é adequada para um problema, pois isto só pode ser determinado empiricamente, mediante a avaliação de várias arquiteturas, há regras na literatura que sugerem o número de neurônios escondidos, tal como a regra da Pirâmide Geométrica, dada a seguir:

$$N_h = \alpha \cdot \sqrt{N_i \cdot N_o},$$

em que $N_h$ é o número de neurônios ocultos (a serem distribuídos em uma ou duas camadas ocultas), $N_i$ é o número de neurônios na camada de entrada e $N_o$ é o número de neurônios na camada de saída. Como $N_h$ deve ser um número inteiro, arredonde o resultado para o inteiro mais próximo.

1. Consulte a documentação da classe MLPClassifier (disponível em https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html) e obtenha os valores de $N_i$ e $N_o$ para a base _Dry Bean_, considerando o número de atributos preditores e o número de classes do problema.
2. Teste os valores de $\alpha$ como sendo iguais a $0.5$, $2$ e $3$.
3. Proponha pelo menos 30 redes neurais segundo a regra da pirâmide geométrica e teste-as nos mesmos termos estabelecidos anteriormente (solver, épocas, 10 repetições com as mesmas partições holdout, escalonamento, etc.)  
    3.1 Apresente as top-3 melhores redes no tocante à F-Score e Acurácia, com base no conjunto de teste

In [ ]:
# Numero de entradas (Ni) e de saidas (No) da MLPClassifier para esta base
N_i = x.shape[1]
N_o = y.nunique()
print('N_i =', N_i, ' N_o =', N_o)


N_i = 16  N_o = 7


In [ ]:
# Regra da Piramide Geometrica: Nh = alpha * sqrt(Ni * No)
# Testamos alpha = 0.5, 2 e 3. Para cada alpha, alem da rede de uma camada com Nh neuronios, geramos variantes de duas camadas distribuindo os Nh neuronios entre elas, de modo a obter pelo menos 30 arquiteturas distintas no total.

arquiteturas_piramide = {}
qtd_splits_por_alpha = {0.5: 4, 2: 12, 3: 11}  # escolhido para resultar em >= 30 arquiteturas unicas

for alpha in [0.5, 2, 3]:
    Nh = round(alpha * math.sqrt(N_i * N_o))
    arquiteturas_piramide[f'P_a{alpha}_1camada'] = (Nh,)
    n_splits = min(qtd_splits_por_alpha[alpha], Nh - 1)
    passo = (Nh - 1) / (n_splits + 1)
    usados = set()
    k = 1
    contador = 0
    while contador < n_splits:
        l1 = max(1, min(Nh - 1, round(passo * k)))
        if l1 not in usados:
            usados.add(l1)
            arquiteturas_piramide[f'P_a{alpha}_split{contador+1}'] = (l1, Nh - l1)
            contador += 1
        k += 1

print('Quantidade de arquiteturas geradas pela pirâmide:', len(arquiteturas_piramide))


Quantidade de arquiteturas geradas pela pirâmide: 30


In [ ]:
# Testamos cada arquitetura da piramide nos mesmos termos da seção anterior

linhas = []
for nome_arq, camadas in arquiteturas_piramide.items():
    for solver in ['adam', 'sgd']:
        for epocas in [100, 150, 200]:
            for p, (tr_idx, te_idx) in enumerate(particoes):
                x_tr, x_te = x.loc[tr_idx], x.loc[te_idx]
                y_tr, y_te = y.loc[tr_idx], y.loc[te_idx]

                escalonador = StandardScaler().fit(x_tr)
                x_tr_s = escalonador.transform(x_tr)
                x_te_s = escalonador.transform(x_te)

                rede = MLPClassifier(hidden_layer_sizes=camadas, activation='relu',
                                      solver=solver, max_iter=epocas, random_state=p)
                rede.fit(x_tr_s, y_tr)
                pred = rede.predict(x_te_s)

                linhas.append({
                    'arquitetura': nome_arq, 'camadas': camadas, 'solver': solver,
                    'epocas': epocas, 'particao': p,
                    'acuracia': accuracy_score(y_te, pred),
                    'f1_macro': f1_score(y_te, pred, average='macro'),
                })

resultados_piramide = pd.DataFrame(linhas)
print('Quantidade de treinamentos realizados:', len(resultados_piramide))


Quantidade de treinamentos realizados: 1800


In [ ]:
resumo_piramide = resultados_piramide.groupby(['arquitetura', 'camadas', 'solver', 'epocas']).agg(
    acuracia_media=('acuracia', 'mean'), acuracia_dp=('acuracia', 'std'),
    f1_media=('f1_macro', 'mean'), f1_dp=('f1_macro', 'std'),
).reset_index()

print('Top-3 arquiteturas da pirâmide por F-Score (média macro):')
print(resumo_piramide.sort_values('f1_media', ascending=False).head(3).to_string(index=False))
print()
print('Top-3 arquiteturas da pirâmide por Acurácia:')
print(resumo_piramide.sort_values('acuracia_media', ascending=False).head(3).to_string(index=False))


Top-3 arquiteturas da pirâmide por F-Score (média macro):
 arquitetura camadas solver  epocas  acuracia_media  acuracia_dp  f1_media    f1_dp
P_a3_1camada   (32,)   adam     200        0.932689     0.004571  0.943451  0.004076
P_a3_split10 (26, 6)   adam     200        0.932199     0.002921  0.943002  0.002821
 P_a3_split9  (23, 9)   adam     200        0.931807     0.004284  0.942653  0.003851

Top-3 arquiteturas da pirâmide por Acurácia:
 arquitetura camadas solver  epocas  acuracia_media  acuracia_dp  f1_media    f1_dp
P_a3_1camada   (32,)   adam     200        0.932689     0.004571  0.943451  0.004076
P_a3_split10 (26, 6)   adam     200        0.932199     0.002921  0.943002  0.002821
 P_a3_split9  (23, 9)   adam     200        0.931807     0.004284  0.942653  0.003851


## Considerações Parciais

1. É possível identificar uma rede com desempenho superior às demais? 
2. Qual estratégia mostrou-se mais produtiva para a obtenção de boas arquiteturas (Estratégia Própria ou Pirâmide Geométrica)? Por quê?
3. Um número maior de épocas trouxe melhorias? Justifique com os resultados obtidos, considerando também cada um dos solvers (Adam e SGD)
4. Qual a maior dificuldade de resolução do problema proposto perante as RNAs?

**1) É possível identificar uma rede com desempenho superior às demais?**

Sim. Dentre todas as arquiteturas e combinações de hiperparâmetros avaliadas, a rede com uma única camada oculta de **64 neurônios**,
treinada com **Adam** e **200 épocas**, obteve o melhor desempenho médio: acurácia de
**0,934 ± 0,004** e F-Score macro de **0,945 ± 0,004**. As demais redes do topo do ranking ficaram muito
próximas, com diferenças de poucos milésimos, dentro da faixa de desvio padrão observada.

**2) Qual estratégia foi mais produtiva, a Estratégia Própria ou a Pirâmide Geométrica?**

As duas estratégias chegaram a resultados muito parecidos: a melhor rede da estratégia própria
(64 neurônios) teve F-Score médio de 0,945, enquanto a melhor rede da pirâmide (32 neurônios, obtida
com alpha = 3) teve F-Score médio de 0,943. A regra da pirâmide, no entanto, foi mais "econômica" ao
indicar diretamente poucos candidatos de tamanho razoável, enquanto na estratégia própria foi necessário escolher os 10
tamanhos por tentativa. Dessa forma, a pirâmide se mostrou uma heurística prática e tão produtiva
quanto a escolha manual, com a vantagem de ser mais simples de aplicar a novos problemas.

**3) Um número maior de épocas trouxe melhorias?**

Sim, de forma consistente. Considerando a média de todas as 2400 execuções realizadas (arquiteturas
próprias + pirâmide), a acurácia média subiu de 0,890 (100 épocas) para 0,897 (150 épocas) e para
0,901 (200 épocas), o mesmo padrão de melhora ocorre separadamente para cada solver. Com Adam, a
acurácia média foi de 0,904 (100 épocas) para 0,910 (200 épocas), com SGD, de 0,877 (100 épocas) para
0,891 (200 épocas). Ou seja, mais épocas ajudaram em ambos os casos, mas o ganho foi proporcionalmente
maior para o SGD, que converge mais lentamente.

**4) Qual a maior dificuldade na resolução do problema?**

A principal dificuldade foi o desbalanceamento das classes:
a classe BOMBAY possui pouco mais de 500 exemplos, enquanto DERMASON possui mais de 3500, o que torna
o F-Score macro mais exigente do que a acurácia simples, pois penaliza igualmente o erro em classes
minoritárias. Outra dificuldade observada foi o alto custo computacional da busca: com 10 arquiteturas
próprias e 30 arquiteturas da pirâmide, cada uma avaliada em 2 solvers x 3 quantidades de épocas x 10
repetições, foram necessários 2400 treinamentos apenas nesta etapa do projeto, o que exige tempo e
paciência mesmo usando redes relativamente pequenas.